In [ ]:
# Google Colab drive mount disabled for local execution
print('Running locally. Using local relative paths.')


In [10]:
YEARS=['2017','2018','2019','2020','2021','2022']

In [11]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict

BASE_PATH = "../WRF-HRRR Computed Dataset/data"
os.listdir("../WRF-HRRR Computed Dataset/data/2017/AL/")

['HRRR_01_AL_2017-01.csv',
 'HRRR_01_AL_2017-08.csv',
 'HRRR_01_AL_2017-04.csv',
 'HRRR_01_AL_2017-05.csv',
 'HRRR_01_AL_2017-06.csv',
 'HRRR_01_AL_2017-07.csv',
 'HRRR_01_AL_2017-09.csv',
 'HRRR_01_AL_2017-12.csv',
 'HRRR_01_AL_2017-03.csv',
 'HRRR_01_AL_2017-02.csv',
 'HRRR_01_AL_2017-11.csv',
 'HRRR_01_AL_2017-10.csv']

In [12]:
from collections import defaultdict

data_dict = defaultdict(lambda: defaultdict(list))

for year in YEARS:
    print(f"Processing {year}")

    year_path = os.path.join(BASE_PATH, year)

    for state in os.listdir(year_path):
        state_path = os.path.join(year_path, state)

        for file in os.listdir(state_path):

            if not file.endswith(".csv"):
                continue

            path = os.path.join(state_path, file)

            df = pd.read_csv(path)

            # ✅ keep only monthly
            df = df[df["Daily/Monthly"] == "Monthly"]

            # ✅ format columns
            df["fips"] = df["FIPS Code"].astype(int).astype(str).str.zfill(5)
            df["month"] = df["Month"].astype(int)

            for _, row in df.iterrows():

                fips = row["fips"]
                month = int(row["month"])

                # 🔥 COMPUTE PER ROW (CORRECT)
                max_temp = row["Max Temperature (K)"]
                min_temp = row["Min Temperature (K)"]
                # 🌡️ Convert to Celsius
                max_temp_C = max_temp - 273.15
                min_temp_C = min_temp - 273.15

                # 🌡️ Avg temp in Celsius
                avg_temp_C = (max_temp_C + min_temp_C) / 2

                # 🔥 Correct GDD
                gdd = max(0, avg_temp_C - 10)

                # 🌡️ Temp range (can stay in K or C — difference same)
                temp_range = max_temp - min_temp

                features = [
                    max_temp_C,                                  # use Celsius
                    min_temp_C,
                    temp_range,
                    gdd,
                    row["Precipitation (kg m**-2)"],
                    row["Relative Humidity (%)"],
                    row["Downward Shortwave Radiation Flux (W m**-2)"],
                    row.get("U Wind Component (m s**-1)", 0),
                    row.get("V Wind Component (m s**-1)", 0),
                    row.get("Surface Pressure (Pa)", 0),
                    row.get("Specific Humidity (kg kg**-1)", 0),
                    row.get("Evapotranspiration (kg m**-2)", 0)
                ]

                data_dict[fips][year].append((month, features))

Processing 2017
Processing 2018
Processing 2019
Processing 2020
Processing 2021
Processing 2022


In [13]:
for fips in data_dict:
    for year in data_dict[fips]:
        seq = data_dict[fips][year]

        # sort Jan → Dec
        seq = sorted(seq, key=lambda x: x[0])

        # keep only features
        data_dict[fips][year] = [x[1] for x in seq]

In [14]:
X_seq_full = []
meta = []

for fips in data_dict:
    for year in data_dict[fips]:

        seq = data_dict[fips][year]

        if len(seq) == 12:  # full year only
            X_seq_full.append(seq)
            meta.append((fips, year))

X_seq_full = np.array(X_seq_full)

print("X_seq_full shape:", X_seq_full.shape)

X_seq_full shape: (17927, 12, 12)


In [15]:
np.savez(
    "../processed/full_weather_9features_v2.npz",
    X_seq_full=X_seq_full,
    meta=np.array(meta)
)

print("Saved successfully 🚀")

Saved successfully 🚀


In [16]:
print(np.isnan(X_seq_full).sum())
print(np.isinf(X_seq_full).sum())

0
0
